# ML-04 — Search Intelligence Data Contract

This notebook defines the data contract for my lane: predicting content decline to prioritize refresh decisions.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. The Contract (Plain Words)

**1. What one row means for my lane:**
One row = one content page on one specific date. Each row captures the page's search performance, engagement metrics, and content attributes for that day.

**2. Which table(s) I'll use:**
- `fact_content_daily_performance` (partitioned by month) for daily metrics
- `dim_content` for content metadata (type, intent, keyword context)
- `dim_clients` for client history coverage

**3. Which time window:**
I'll use a mid-panel month (2026-03) for feature development, with the label defined on the following month (April 2026). The feature window will be the 30 days prior to prediction, and the label window will be the 30 days after.

**4. What I'd predict or rank (label or proxy):**
I'll predict whether a content page's impressions will decline by more than 20% month-over-month. This is a binary classification task where:
- Label = 1 if impressions decline > 20% from previous month
- Label = 0 otherwise

**5. One thing I deliberately exclude:**
I exclude `gsc_avg_position` (search ranking position) because it can vary wildly day-to-day and doesn't reliably indicate content quality or refresh need. A page can rank #1 for one query and #50 for another on the same day.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Connect to Hugging Face dataset
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    print('WARNING: HF_TOKEN not found. Set it as an environment variable or Colab Secret.')
    print('For Colab: Add HF_TOKEN to the Secrets panel (key icon on the left).')

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
else:
    print('Skipping Hugging Face connection - token not available.')

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

if HF_TOKEN:
    print('Connected to FlyRank warehouse')
else:
    print('Notebook ready - set HF_TOKEN to run queries against the warehouse.')

Connected to FlyRank warehouse


## 2. Verify the Contract with Queries

Three verification queries on a mid-panel month (2026-03):

In [2]:
# Only run queries if HF_TOKEN is available
if not HF_TOKEN:
    print('Skipping queries - HF_TOKEN not set. Set it to run against the warehouse.')
else:
    # Query 1: Verify the grain - one row really is one page-date
    grain_check = con.sql(f"""
    SELECT 
        client_hash_id, 
        content_hash_id, 
        report_date,
        COUNT(*) as row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
    """).df()
    
    print('Grain check (should be empty):', grain_check)
    print('Grain holds: one row per page-date')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (should be empty): Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []
Grain holds: one row per page-date


In [3]:
if not HF_TOKEN:
    print('Skipping query - HF_TOKEN not set.')
else:
    # Query 2: Row count and date span for March 2026
    count_span = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        MIN(report_date) as earliest_date,
        MAX(report_date) as latest_date,
        COUNT(DISTINCT client_hash_id) as unique_clients,
        COUNT(DISTINCT content_hash_id) as unique_content
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    """).df()
    
    print('March 2026 statistics:')
    print(count_span)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 statistics:
   total_rows earliest_date latest_date  unique_clients  unique_content
0     9841378    2026-03-01  2026-03-31              55          331437


In [4]:
if not HF_TOKEN:
    print('Skipping query - HF_TOKEN not set.')
else:
    # Query 3: Availability check with IS TRUE filter
    availability = con.sql(f"""
    SELECT 
        ga4_data_available,
        COUNT(*) as row_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY ga4_data_available
    """).df()
    
    print('GA4 data availability in March 2026:')
    print(availability)
    
    # Show rows that survive IS TRUE filter
    surviving_rows = con.sql(f"""
    SELECT COUNT(*) as rows_with_ga4_data
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND ga4_data_available IS TRUE
    """).df()
    
    print('\nRows surviving IS TRUE filter:')
    print(surviving_rows)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 data availability in March 2026:
   ga4_data_available  row_count  percentage
0               False    6408671       65.12
1                <NA>    3018741       30.67
2                True     413966        4.21


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows surviving IS TRUE filter:
   rows_with_ga4_data
0              413966


## 3. Build Features for My Lane

Five features, each with availability justification:

In [5]:
if not HF_TOKEN:
    print('Skipping feature building - HF_TOKEN not set.')
else:
    # Build feature frame for March 2026
    feature_frame = con.sql(f"""
    WITH monthly_stats AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            ga4_sessions,
            ga4_users,
            ga4_engaged_sessions
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
          AND ga4_data_available IS TRUE
    ),
    features AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            -- Feature 1: Total impressions in March (knowable because it's historical data)
            SUM(gsc_impressions) as total_impressions_march,
            -- Feature 2: Total clicks in March (knowable because it's historical data)
            SUM(gsc_clicks) as total_clicks_march,
            -- Feature 3: Click-through rate (computed from features 1 & 2)
            CASE WHEN SUM(gsc_impressions) > 0 
                 THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions) 
                 ELSE 0 END as ctr_march,
            -- Feature 4: Total sessions in March (knowable because it's historical data)
            SUM(ga4_sessions) as total_sessions_march,
            -- Feature 5: Engagement rate (computed from sessions data)
            CASE WHEN SUM(ga4_sessions) > 0 
                 THEN SUM(ga4_engaged_sessions) * 100.0 / SUM(ga4_sessions) 
                 ELSE 0 END as engagement_rate_march
        FROM monthly_stats
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT * FROM features
    """).df()
    
    print('Feature frame shape:', feature_frame.shape)
    print('\nFeatures built:')
    print(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (90489, 7)

Features built:
            client_hash_id           content_hash_id  total_impressions_march  \
0  client_23a62021009f63c4  content_fe7321017e85c4c1                  13044.0   
1  client_23a62021009f63c4  content_496ecabe1eae48bb                  27474.0   
2  client_23a62021009f63c4  content_5d81736ef7aeb0d7                    209.0   
3  client_23a62021009f63c4  content_3515e841687db344                   2700.0   
4  client_23a62021009f63c4  content_fba239f8996abcab                    976.0   

   total_clicks_march  ctr_march  total_sessions_march  engagement_rate_march  
0                28.0   0.214658                  81.0               2.469136  
1                69.0   0.251147                 189.0               1.587302  
2                 0.0   0.000000                   7.0               0.000000  
3                13.0   0.481481                  25.0               4.000000  
4                25.0   2.561475                  52.0          

**Feature Availability Notes:**

1. **total_impressions_march** - Knowable at decision moment because it's historical GSC data from March 2026
2. **total_clicks_march** - Knowable at decision moment because it's historical GSC data from March 2026
3. **ctr_march** - Knowable at decision moment because it's computed from impressions and clicks (both historical)
4. **total_sessions_march** - Knowable at decision moment because it's historical GA4 data from March 2026
5. **engagement_rate_march** - Knowable at decision moment because it's computed from session data (both historical)

All features are from March 2026, which is before the prediction moment (April 2026).

In [6]:
if not HF_TOKEN:
    print('Skipping label creation - HF_TOKEN not set.')
else:
    # Create label for April 2026 (the month we're predicting)
    label_data = con.sql(f"""
    WITH march_stats AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) as impressions_march
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    april_stats AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) as impressions_april
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT 
        m.client_hash_id,
        m.content_hash_id,
        m.impressions_march,
        COALESCE(a.impressions_april, 0) as impressions_april,
        -- Label: 1 if impressions decline > 20%, 0 otherwise
        CASE 
            WHEN m.impressions_march > 0 AND 
                 COALESCE(a.impressions_april, 0) < 0.8 * m.impressions_march 
            THEN 1 
            ELSE 0 
        END as is_declining
    FROM march_stats m
    LEFT JOIN april_stats a 
        ON m.client_hash_id = a.client_hash_id 
        AND m.content_hash_id = a.content_hash_id
    """).df()
    
    # Merge features with labels
    model_data = feature_frame.merge(label_data, on=['client_hash_id', 'content_hash_id'], how='inner')
    
    print('Model data shape:', model_data.shape)
    print('\nLabel distribution:')
    print(model_data['is_declining'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model data shape: (90489, 10)

Label distribution:
is_declining
0    55028
1    35461
Name: count, dtype: int64


## 4. The Leakage Experiment

Now I'll demonstrate the trap: adding a label-derived column and watching the score jump.

In [7]:
if not HF_TOKEN:
    print('Skipping model training - HF_TOKEN not set.')
else:
    # First, train honest model without leakage
    feature_cols = ['total_impressions_march', 'total_clicks_march', 'ctr_march', 
                    'total_sessions_march', 'engagement_rate_march']
    
    X = model_data[feature_cols].fillna(0)
    y = model_data['is_declining']
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    
    # Honest model
    honest_model = RandomForestClassifier(n_estimators=100, random_state=42)
    honest_model.fit(X_train, y_train)
    honest_score = honest_model.score(X_test, y_test)
    
    print('Honest model accuracy (no leakage):', round(honest_score, 4))

Honest model accuracy (no leakage): 0.6351


In [8]:
if not HF_TOKEN:
    print('Skipping leaky model - HF_TOKEN not set.')
else:
    # Now add the LEAKAGE column - impressions_april is the label window!
    # This is cheating because we're using future information to predict the future
    model_data_leaky = model_data.copy()
    model_data_leaky['leaky_feature'] = model_data_leaky['impressions_april']  # LEAKAGE!
    
    feature_cols_leaky = feature_cols + ['leaky_feature']
    X_leaky = model_data_leaky[feature_cols_leaky].fillna(0)
    
    # Train/test split with leakage
    X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
    
    # Leaky model
    leaky_model = RandomForestClassifier(n_estimators=100, random_state=42)
    leaky_model.fit(X_train_l, y_train_l)
    leaky_score = leaky_model.score(X_test_l, y_test_l)
    
    print('Leaky model accuracy (with future info):', round(leaky_score, 4))
    print('\nScore jump:', round(leaky_score - honest_score, 4), 'points')
    print('This is the trap - the model learns to cheat by seeing the future!')

Leaky model accuracy (with future info): 0.7897

Score jump: 0.1546 points
This is the trap - the model learns to cheat by seeing the future!


In [9]:
if not HF_TOKEN:
    print('Skipping cleanup - HF_TOKEN not set.')
else:
    # DELETE the leaky column and keep the honest number
    model_data_clean = model_data.drop(columns=['impressions_april'])
    
    print('Leaky column deleted.')
    print('Keeping honest model accuracy:', round(honest_score, 4))
    print('\nLesson learned: Never use future information as a feature!')

Leaky column deleted.
Keeping honest model accuracy: 0.6351

Lesson learned: Never use future information as a feature!


## 5. One Named Limitation

**Limitation: Unbalanced Panel History**

My slice assumes all clients have similar history depth, but the data shows clients have wildly different `gsc_data_start` dates. Some clients have 17 months of history, others only 3 months. This means:

- Clients with shorter history may have less reliable features
- The model may perform better on clients with more data
- Future work should use per-client windows instead of global calendar windows

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.